## Install Packages

In [1]:
import subprocess
import sys

def install_packages():
    packages = [
        "transformers>=4.36.0",
        "datasets>=2.16.0",
        "peft>=0.7.0",
        "accelerate>=0.25.0",
        "evaluate>=0.4.0",
        "rouge-score>=0.1.2",
        "nltk>=3.8.0",
        "sentencepiece>=0.1.99",
        "protobuf>=4.25.0",
        "scikit-learn>=1.3.0",
        "matplotlib>=3.8.0",
        "seaborn>=0.13.0",
    ]
    for pkg in packages:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
    print("All packages installed successfully.")

install_packages()

All packages installed successfully.


## Imports and Configuration

In [2]:
import os
import time
import json
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import nltk

from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model, TaskType
import evaluate

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

CONFIG = {
    "model_name": "google/flan-t5-base",
    "dataset_name": "keivalya/MedQuad-MedicalQnADataset",
    "max_input_length": 512,
    "max_target_length": 256,
    "train_split": 0.8,
    "val_split": 0.1,
    "test_split": 0.1,
    "batch_size": 8,
    "learning_rate": 3e-4,
    "num_epochs": 5,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.1,
    "output_dir": "./results",
    "seed": SEED,
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

Using device: cpu


## Dataset Loading and Splits

In [3]:
dataset = load_dataset(CONFIG["dataset_name"])
print(f"\nDataset: {CONFIG['dataset_name']}, Total samples: {len(dataset['train'])}")
print(f"Sample Q: {dataset['train'][0]['Question'][:100]}...")
print(f"Sample A: {dataset['train'][0]['Answer'][:100]}...")

full_dataset = dataset["train"].shuffle(seed=SEED)
total_size = len(full_dataset)
train_size = int(total_size * CONFIG["train_split"])
val_size = int(total_size * CONFIG["val_split"])

split_dataset = DatasetDict({
    "train": full_dataset.select(range(train_size)),
    "validation": full_dataset.select(range(train_size, train_size + val_size)),
    "test": full_dataset.select(range(train_size + val_size, total_size)),
})

print(f"Train: {len(split_dataset['train'])}, Val: {len(split_dataset['validation'])}, Test: {len(split_dataset['test'])}")


Dataset: keivalya/MedQuad-MedicalQnADataset, Total samples: 16407
Sample Q: Who is at risk for Lymphocytic Choriomeningitis (LCM)? ?...
Sample A: LCMV infections can occur after exposure to fresh urine, droppings, saliva, or nesting materials fro...
Train: 13125, Val: 1640, Test: 1642


## Tokenization

In [4]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])

def preprocess_function(examples):
    inputs = [f"Answer the following medical question: {q}" for q in examples["Question"]]
    targets = examples["Answer"]

    model_inputs = tokenizer(inputs, max_length=CONFIG["max_input_length"], truncation=True, padding="max_length")
    labels = tokenizer(targets, max_length=CONFIG["max_target_length"], truncation=True, padding="max_length")
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing datasets...")
tokenized_datasets = split_dataset.map(
    preprocess_function, batched=True,
    remove_columns=split_dataset["train"].column_names, desc="Tokenizing",
)
print("Tokenization complete.")

Tokenizing datasets...


Tokenizing:   0%|          | 0/13125 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/1640 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/1642 [00:00<?, ? examples/s]

Tokenization complete.


## Metrics Setup

In [5]:
rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    rouge_results = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)

    bleu_scores = []
    for pred, ref in zip(decoded_preds, decoded_labels):
        try:
            pt, rt = pred.split(), ref.split()
            if pt and rt:
                bleu_scores.append(bleu_metric.compute(predictions=[pt], references=[[rt]])["bleu"])
            else:
                bleu_scores.append(0.0)
        except Exception:
            bleu_scores.append(0.0)

    return {k: round(v, 4) for k, v in {
        "rouge1": rouge_results["rouge1"], "rouge2": rouge_results["rouge2"],
        "rougeL": rouge_results["rougeL"], "bleu": np.mean(bleu_scores),
    }.items()}

## Baseline Evaluation (Zero-Shot)

In [6]:
print("Loading baseline model...")
baseline_model = AutoModelForSeq2SeqLM.from_pretrained(CONFIG["model_name"])
baseline_model.to(device)
model_params = sum(p.numel() for p in baseline_model.parameters())
print(f"Model parameters: {model_params:,} ({model_params/1e6:.1f}M)")

print("Evaluating baseline (zero-shot) on test set...")
baseline_start_time = time.time()
baseline_predictions, baseline_references, test_questions = [], [], []

baseline_model.eval()
with torch.no_grad():
    for i, sample in enumerate(split_dataset["test"]):
        input_text = f"Answer the following medical question: {sample['Question']}"
        inputs = tokenizer(input_text, return_tensors="pt", max_length=CONFIG["max_input_length"], truncation=True).to(device)
        outputs = baseline_model.generate(**inputs, max_new_tokens=CONFIG["max_target_length"], num_beams=4, early_stopping=True)
        baseline_predictions.append(tokenizer.decode(outputs[0], skip_special_tokens=True).strip())
        baseline_references.append(sample["Answer"].strip())
        test_questions.append(sample["Question"])
        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{len(split_dataset['test'])} samples...")

baseline_eval_time = time.time() - baseline_start_time

baseline_rouge = rouge_metric.compute(predictions=baseline_predictions, references=baseline_references, use_stemmer=True)
baseline_bleu_scores = []
for pred, ref in zip(baseline_predictions, baseline_references):
    try:
        pt, rt = pred.split(), ref.split()
        if pt and rt:
            baseline_bleu_scores.append(bleu_metric.compute(predictions=[pt], references=[[rt]])["bleu"])
        else:
            baseline_bleu_scores.append(0.0)
    except Exception:
        baseline_bleu_scores.append(0.0)

baseline_metrics = {
    "rouge1": round(baseline_rouge["rouge1"], 4), "rouge2": round(baseline_rouge["rouge2"], 4),
    "rougeL": round(baseline_rouge["rougeL"], 4), "bleu": round(np.mean(baseline_bleu_scores), 4),
}

print(f"\nBaseline Results: ROUGE-1={baseline_metrics['rouge1']:.4f}, ROUGE-2={baseline_metrics['rouge2']:.4f}, ROUGE-L={baseline_metrics['rougeL']:.4f}, BLEU={baseline_metrics['bleu']:.4f}")

for i in range(min(3, len(test_questions))):
    print(f"\nQ: {test_questions[i][:150]}...")
    print(f"Baseline: {baseline_predictions[i][:150]}...")
    print(f"Reference: {baseline_references[i][:150]}...")

del baseline_model
torch.cuda.empty_cache() if torch.cuda.is_available() else None

Loading baseline model...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model parameters: 247,577,856 (247.6M)
Evaluating baseline (zero-shot) on test set...


  50/1642 samples...


## Hardware Assessment + LoRA Setup

In [ ]:
print(f"""
Hardware Assessment:
  Full fine-tuning Flan-T5-Base (248M params) requires ~8-12 GB GPU memory.
  Available hardware: AMD Ryzen 5 5625U, no dedicated GPU - INFEASIBLE.
  Solution: LoRA reduces trainable parameters by ~99%, needing only ~2-3 GB.
  Running on Google Colab (T4 GPU, 15GB VRAM) for LoRA fine-tuning.
""")

print("Loading model for LoRA fine-tuning...")
model = AutoModelForSeq2SeqLM.from_pretrained(CONFIG["model_name"])

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=["q", "v"], bias="none",
)

model = get_peft_model(model, lora_config)
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
reduction = (1 - trainable_params / total_params) * 100

print(f"LoRA Config: r={CONFIG['lora_r']}, alpha={CONFIG['lora_alpha']}, targets=[q, v]")
print(f"Total params: {total_params:,}, Trainable: {trainable_params:,} ({trainable_params/total_params*100:.4f}%), Reduction: {reduction:.2f}%")

## Training

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)

training_args = Seq2SeqTrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    learning_rate=CONFIG["learning_rate"],
    weight_decay=0.01, warmup_steps=100,
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="rougeL", greater_is_better=True,
    predict_with_generate=True,
    generation_max_length=CONFIG["max_target_length"],
    logging_dir=f"{CONFIG['output_dir']}/logs", logging_steps=50,
    report_to="none", seed=SEED,
    fp16=torch.cuda.is_available(), dataloader_num_workers=0,
    optim="adamw_torch",
)

trainer = Seq2SeqTrainer(
    model=model, args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer, data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print(f"\nTraining: {CONFIG['num_epochs']} epochs, batch_size={CONFIG['batch_size']}, lr={CONFIG['learning_rate']}, fp16={torch.cuda.is_available()}")
train_start_time = time.time()
train_result = trainer.train()
train_time = time.time() - train_start_time
print(f"Training complete: {train_time:.1f}s ({train_time/60:.1f} min), loss={train_result.training_loss:.4f}")

training_history = trainer.state.log_history
epoch_metrics = []
for entry in training_history:
    if "eval_rouge1" in entry:
        epoch_metrics.append({
            "epoch": entry.get("epoch", 0), "eval_loss": entry.get("eval_loss", 0),
            "eval_rouge1": entry.get("eval_rouge1", 0), "eval_rouge2": entry.get("eval_rouge2", 0),
            "eval_rougeL": entry.get("eval_rougeL", 0), "eval_bleu": entry.get("eval_bleu", 0),
        })

for em in epoch_metrics:
    print(f"  Epoch {em['epoch']:.0f}: Loss={em['eval_loss']:.4f}, R1={em['eval_rouge1']:.4f}, R2={em['eval_rouge2']:.4f}, RL={em['eval_rougeL']:.4f}, BLEU={em['eval_bleu']:.4f}")

## LoRA Evaluation on Test Set

In [ ]:
print("\nEvaluating LoRA model on test set...")
lora_start_time = time.time()
lora_predictions, lora_references = [], []

model.eval()
with torch.no_grad():
    for i, sample in enumerate(split_dataset["test"]):
        input_text = f"Answer the following medical question: {sample['Question']}"
        inputs = tokenizer(input_text, return_tensors="pt", max_length=CONFIG["max_input_length"], truncation=True).to(device)
        outputs = model.generate(**inputs, max_new_tokens=CONFIG["max_target_length"], num_beams=4, early_stopping=True)
        lora_predictions.append(tokenizer.decode(outputs[0], skip_special_tokens=True).strip())
        lora_references.append(sample["Answer"].strip())
        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{len(split_dataset['test'])} samples...")

lora_eval_time = time.time() - lora_start_time

lora_rouge = rouge_metric.compute(predictions=lora_predictions, references=lora_references, use_stemmer=True)
lora_bleu_scores = []
for pred, ref in zip(lora_predictions, lora_references):
    try:
        pt, rt = pred.split(), ref.split()
        if pt and rt:
            lora_bleu_scores.append(bleu_metric.compute(predictions=[pt], references=[[rt]])["bleu"])
        else:
            lora_bleu_scores.append(0.0)
    except Exception:
        lora_bleu_scores.append(0.0)

lora_metrics = {
    "rouge1": round(lora_rouge["rouge1"], 4), "rouge2": round(lora_rouge["rouge2"], 4),
    "rougeL": round(lora_rouge["rougeL"], 4), "bleu": round(np.mean(lora_bleu_scores), 4),
}

print(f"\nLoRA Results: ROUGE-1={lora_metrics['rouge1']:.4f}, ROUGE-2={lora_metrics['rouge2']:.4f}, ROUGE-L={lora_metrics['rougeL']:.4f}, BLEU={lora_metrics['bleu']:.4f}")

for i in range(min(3, len(test_questions))):
    print(f"\nQ: {test_questions[i][:150]}...")
    print(f"LoRA: {lora_predictions[i][:150]}...")
    print(f"Reference: {lora_references[i][:150]}...")

## Comparative Analysis

In [ ]:
comparison_data = {
    "Metric": ["ROUGE-1", "ROUGE-2", "ROUGE-L", "BLEU"],
    "Baseline (Zero-Shot)": [baseline_metrics["rouge1"], baseline_metrics["rouge2"], baseline_metrics["rougeL"], baseline_metrics["bleu"]],
    "LoRA Fine-Tuned": [lora_metrics["rouge1"], lora_metrics["rouge2"], lora_metrics["rougeL"], lora_metrics["bleu"]],
}

comparison_df = pd.DataFrame(comparison_data)
comparison_df["Improvement"] = comparison_df["LoRA Fine-Tuned"] - comparison_df["Baseline (Zero-Shot)"]
comparison_df["Improvement (%)"] = ((comparison_df["Improvement"] / comparison_df["Baseline (Zero-Shot)"].replace(0, 1)) * 100).round(2)

print("\n" + comparison_df.to_string(index=False))
print(f"\nTotal params: {total_params:,}, Trainable (LoRA): {trainable_params:,}, Reduction: {reduction:.2f}%")
print(f"Baseline eval: {baseline_eval_time:.1f}s, LoRA eval: {lora_eval_time:.1f}s, Training: {train_time:.1f}s")

## Visualization

In [ ]:
os.makedirs(CONFIG["output_dir"], exist_ok=True)

fig, ax = plt.subplots(figsize=(10, 6))
metrics_names = ["ROUGE-1", "ROUGE-2", "ROUGE-L", "BLEU"]
baseline_values = [baseline_metrics["rouge1"], baseline_metrics["rouge2"], baseline_metrics["rougeL"], baseline_metrics["bleu"]]
lora_values = [lora_metrics["rouge1"], lora_metrics["rouge2"], lora_metrics["rougeL"], lora_metrics["bleu"]]
x = np.arange(len(metrics_names))
width = 0.35
bars1 = ax.bar(x - width/2, baseline_values, width, label="Baseline (Zero-Shot)", color="#3498db", edgecolor="black", linewidth=0.5)
bars2 = ax.bar(x + width/2, lora_values, width, label="LoRA Fine-Tuned", color="#e74c3c", edgecolor="black", linewidth=0.5)
ax.set_xlabel("Evaluation Metric", fontsize=12)
ax.set_ylabel("Score", fontsize=12)
ax.set_title("Baseline vs LoRA Fine-Tuned: Medical Q&A Performance", fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(metrics_names, fontsize=11)
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3)
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005, f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005, f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/metric_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

## More Visualizations

In [ ]:
if epoch_metrics:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    epochs = [em["epoch"] for em in epoch_metrics]
    axes[0].plot(epochs, [em["eval_loss"] for em in epoch_metrics], "b-o", linewidth=2, markersize=8, label="Validation Loss")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].set_title("Validation Loss"); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(epochs, [em["eval_rouge1"] for em in epoch_metrics], "r-o", linewidth=2, markersize=8, label="ROUGE-1")
    axes[1].plot(epochs, [em["eval_rouge2"] for em in epoch_metrics], "g-s", linewidth=2, markersize=8, label="ROUGE-2")
    axes[1].plot(epochs, [em["eval_rougeL"] for em in epoch_metrics], "b-^", linewidth=2, markersize=8, label="ROUGE-L")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Score"); axes[1].set_title("ROUGE Scores"); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{CONFIG['output_dir']}/training_curves.png", dpi=300, bbox_inches="tight")
    plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(["Total\nParameters", "Trainable\n(LoRA)"], [total_params, trainable_params], color=["#3498db", "#e74c3c"], edgecolor="black")
ax.set_ylabel("Parameters"); ax.set_title("Parameter Efficiency: LoRA vs Full Model"); ax.set_yscale("log"); ax.grid(axis="y", alpha=0.3)
for bar, val in zip(bars, [total_params, trainable_params]):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() * 1.1, f"{val:,}", ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/parameter_efficiency.png", dpi=300, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
N = len(metrics_names)
angles = [n / float(N) * 2 * np.pi for n in range(N)] + [0]
ax.plot(angles, baseline_values + baseline_values[:1], "o-", linewidth=2, label="Baseline", color="#3498db")
ax.fill(angles, baseline_values + baseline_values[:1], alpha=0.15, color="#3498db")
ax.plot(angles, lora_values + lora_values[:1], "o-", linewidth=2, label="LoRA Fine-Tuned", color="#e74c3c")
ax.fill(angles, lora_values + lora_values[:1], alpha=0.15, color="#e74c3c")
ax.set_xticks(angles[:-1]); ax.set_xticklabels(metrics_names, fontsize=12)
ax.set_title("Performance Radar: Baseline vs LoRA", fontsize=14, pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=11)
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/radar_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import seaborn as sns

# ============================================================
# Figure 1: Improvement Analysis (Absolute + Percentage)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

imp_vals = [lo - bl for lo, bl in zip(lora_values, baseline_values)]
colors_imp = ["#2ecc71" if v >= 0 else "#e74c3c" for v in imp_vals]
axes[0].bar(metrics_names, imp_vals, color=colors_imp, edgecolor="black", linewidth=0.5)
axes[0].axhline(y=0, color="black", linewidth=0.8, linestyle="--")
for i, v in enumerate(imp_vals):
    axes[0].text(i, v + (0.005 if v >= 0 else -0.02), f"{v:+.4f}", ha="center", fontsize=9, fontweight="bold")
axes[0].set_title("Absolute Improvement (LoRA - Baseline)", fontsize=11)
axes[0].set_ylabel("Score Difference"); axes[0].grid(axis="y", alpha=0.3)

pct_vals = [(lo - bl) / bl * 100 if bl != 0 else 0 for lo, bl in zip(lora_values, baseline_values)]
colors_pct = ["#2ecc71" if v >= 0 else "#e74c3c" for v in pct_vals]
axes[1].bar(metrics_names, pct_vals, color=colors_pct, edgecolor="black", linewidth=0.5)
axes[1].axhline(y=0, color="black", linewidth=0.8, linestyle="--")
for i, v in enumerate(pct_vals):
    axes[1].text(i, v + (2 if v >= 0 else -5), f"{v:+.1f}%", ha="center", fontsize=9, fontweight="bold")
axes[1].set_title("Percentage Improvement", fontsize=11)
axes[1].set_ylabel("Improvement (%)"); axes[1].grid(axis="y", alpha=0.3)

plt.suptitle("Flan-T5-Base Medical Q&A: Improvement Analysis", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/improvement_analysis.png", dpi=200, bbox_inches="tight")
plt.show()

# ============================================================
# Figure 2: Training Dynamics (Step-Level Loss + Epoch Metrics)
# ============================================================
train_steps = [e for e in training_history if "loss" in e and "eval_loss" not in e]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if train_steps:
    steps = [e["step"] for e in train_steps]
    losses = [e["loss"] for e in train_steps]
    axes[0].plot(steps, losses, color="#e74c3c", linewidth=1.0, alpha=0.5, label="Raw Loss")
    if len(losses) > 5:
        w = min(10, max(3, len(losses)//3))
        ma = np.convolve(losses, np.ones(w)/w, mode="valid")
        axes[0].plot(steps[w-1:], ma, color="#2c3e50", linewidth=2.5, label=f"Moving Avg (w={w})")
    axes[0].set_title("Training Loss per Step"); axes[0].set_xlabel("Step"); axes[0].set_ylabel("Loss")
    axes[0].legend(); axes[0].grid(alpha=0.3)

if epoch_metrics:
    eps = [e["epoch"] for e in epoch_metrics]
    ax2 = axes[1].twinx()
    axes[1].plot(eps, [e.get("eval_rouge1", 0) for e in epoch_metrics], "r-o", lw=2, markersize=7, label="ROUGE-1")
    axes[1].plot(eps, [e.get("eval_rougeL", 0) for e in epoch_metrics], "b-^", lw=2, markersize=7, label="ROUGE-L")
    axes[1].plot(eps, [e.get("eval_bleu", 0) for e in epoch_metrics], "m-D", lw=2, markersize=7, label="BLEU")
    ax2.plot(eps, [e["eval_loss"] for e in epoch_metrics], "k--o", lw=1.5, markersize=5, alpha=0.5, label="Eval Loss")
    ax2.set_ylabel("Eval Loss")
    axes[1].set_ylabel("Score"); axes[1].set_xlabel("Epoch")
    axes[1].set_title("Validation Metrics Across Epochs")
    h1, l1 = axes[1].get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    axes[1].legend(h1 + h2, l1 + l2, loc="center right", fontsize=8)
    axes[1].grid(alpha=0.3)

plt.suptitle("Flan-T5-Base Medical Q&A: Training Dynamics", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/training_dynamics.png", dpi=200, bbox_inches="tight")
plt.show()

# ============================================================
# Figure 3: Prediction Analysis (Length Distribution + BLEU Box)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bl_lens = [len(p.split()) for p in baseline_predictions]
lo_lens = [len(p.split()) for p in lora_predictions]
ref_lens = [len(r.split()) for r in baseline_references]

axes[0].hist(ref_lens, bins=20, alpha=0.4, label=f"Reference (mean={np.mean(ref_lens):.1f})", color="#2ecc71", edgecolor="black")
axes[0].hist(bl_lens, bins=20, alpha=0.5, label=f"Baseline (mean={np.mean(bl_lens):.1f})", color="#3498db", edgecolor="black")
axes[0].hist(lo_lens, bins=20, alpha=0.5, label=f"LoRA (mean={np.mean(lo_lens):.1f})", color="#e74c3c", edgecolor="black")
axes[0].set_title("Prediction Length Distribution"); axes[0].set_xlabel("Words"); axes[0].set_ylabel("Frequency")
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

bp = axes[1].boxplot([baseline_bleu_scores, lora_bleu_scores], labels=["Baseline", "LoRA"], patch_artist=True,
                      medianprops=dict(color="black", linewidth=2))
bp["boxes"][0].set_facecolor("#3498db"); bp["boxes"][0].set_alpha(0.6)
bp["boxes"][1].set_facecolor("#e74c3c"); bp["boxes"][1].set_alpha(0.6)
axes[1].set_title("Per-Sample BLEU Distribution"); axes[1].set_ylabel("BLEU Score"); axes[1].grid(alpha=0.3)

plt.suptitle("Flan-T5-Base Medical Q&A: Prediction Analysis", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/prediction_analysis.png", dpi=200, bbox_inches="tight")
plt.show()

# ============================================================
# Figure 4: Extended Dashboard (Pie + Timing + Heatmap + CDF)
# ============================================================
fig = plt.figure(figsize=(16, 12))

ax1 = fig.add_subplot(2, 2, 1)
frozen = total_params - trainable_params
ax1.pie([trainable_params, frozen], labels=[f"Trainable\n({trainable_params:,})", f"Frozen\n({frozen:,})"],
        colors=["#e74c3c", "#3498db"], autopct="%1.2f%%", startangle=90,
        textprops={"fontsize": 9}, explode=[0.05, 0])
ax1.set_title("LoRA Parameter Distribution", fontsize=12)

ax2 = fig.add_subplot(2, 2, 2)
times = [baseline_eval_time, train_time, lora_eval_time]
time_labels = ["Baseline\nEval", "LoRA\nTraining", "LoRA\nEval"]
bars = ax2.bar(time_labels, times, color=["#3498db", "#e74c3c", "#2ecc71"], edgecolor="black", linewidth=0.5)
for bar, t in zip(bars, times):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + max(times)*0.02,
             f"{t:.1f}s", ha="center", fontsize=10, fontweight="bold")
ax2.set_title("Timing Comparison"); ax2.set_ylabel("Time (seconds)"); ax2.grid(axis="y", alpha=0.3)

ax3 = fig.add_subplot(2, 2, 3)
heatmap_data = np.array([baseline_values, lora_values])
sns.heatmap(heatmap_data, annot=True, fmt=".4f", xticklabels=metrics_names,
            yticklabels=["Baseline", "LoRA"], cmap="YlOrRd", ax=ax3,
            linewidths=0.5, linecolor="black", cbar_kws={"shrink": 0.8})
ax3.set_title("Metrics Heatmap", fontsize=12)

ax4 = fig.add_subplot(2, 2, 4)
sorted_bl = np.sort(baseline_bleu_scores)
sorted_lo = np.sort(lora_bleu_scores)
ax4.plot(sorted_bl, np.linspace(0, 1, len(sorted_bl)), lw=2, label="Baseline", color="#3498db")
ax4.plot(sorted_lo, np.linspace(0, 1, len(sorted_lo)), lw=2, label="LoRA", color="#e74c3c")
ax4.set_title("Cumulative BLEU Distribution (CDF)"); ax4.set_xlabel("BLEU Score")
ax4.set_ylabel("Cumulative Proportion"); ax4.legend(); ax4.grid(alpha=0.3)

plt.suptitle("Flan-T5-Base Medical Q&A: Extended Dashboard", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/extended_dashboard.png", dpi=200, bbox_inches="tight")
plt.show()

# ============================================================
# Figure 5: Per-Sample BLEU Scatter (Baseline vs LoRA)
# ============================================================
fig, ax = plt.subplots(figsize=(8, 8))
min_len = min(len(baseline_bleu_scores), len(lora_bleu_scores))
ax.scatter(baseline_bleu_scores[:min_len], lora_bleu_scores[:min_len], alpha=0.4, s=25, color="#8e44ad")
max_v = max(max(baseline_bleu_scores[:min_len]) if baseline_bleu_scores[:min_len] else 0.01,
            max(lora_bleu_scores[:min_len]) if lora_bleu_scores[:min_len] else 0.01, 0.01)
ax.plot([0, max_v], [0, max_v], "k--", lw=1.5, label="y=x (equal performance)")
ax.set_title("Per-Sample BLEU: Baseline vs LoRA", fontsize=13, fontweight="bold")
ax.set_xlabel("Baseline BLEU", fontsize=11); ax.set_ylabel("LoRA BLEU", fontsize=11)
ax.legend(fontsize=10); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/bleu_scatter.png", dpi=200, bbox_inches="tight")
plt.show()

print(f"All extended visualizations saved to {CONFIG['output_dir']}/")

## Save Results

In [ ]:
results = {
    "config": CONFIG, "baseline_metrics": baseline_metrics, "lora_metrics": lora_metrics,
    "training_time_seconds": train_time, "baseline_eval_time_seconds": baseline_eval_time,
    "lora_eval_time_seconds": lora_eval_time, "total_parameters": total_params,
    "trainable_parameters": trainable_params, "parameter_reduction_percent": round(reduction, 2),
    "epoch_metrics": epoch_metrics,
    "hardware": {"device": str(device), "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"},
}

with open(f"{CONFIG['output_dir']}/results.json", "w") as f:
    json.dump(results, f, indent=2)

comparison_df.to_csv(f"{CONFIG['output_dir']}/comparison.csv", index=False)
model.save_pretrained(f"{CONFIG['output_dir']}/lora_model")
tokenizer.save_pretrained(f"{CONFIG['output_dir']}/lora_model")

examples_df = pd.DataFrame({
    "Question": test_questions[:20], "Baseline_Prediction": baseline_predictions[:20],
    "LoRA_Prediction": lora_predictions[:20], "Reference": baseline_references[:20],
})
examples_df.to_csv(f"{CONFIG['output_dir']}/example_predictions.csv", index=False)

print(f"\nAll results saved to {CONFIG['output_dir']}/")

## Final Summary

In [ ]:
print(f"Model: {CONFIG['model_name']}, Dataset: {CONFIG['dataset_name']}")
print(f"LoRA: r={CONFIG['lora_r']}, alpha={CONFIG['lora_alpha']}, trainable={trainable_params:,} ({trainable_params/total_params*100:.4f}%)")
print(f"Training: {train_time/60:.1f} min on {device}")